# HEART HPC validation — step by step (CMRI)

Interactive, cell-by-cell checks of the HEART pretraining stack on CMRI, before and after the 30-minute GPU smoke test.

* Run the cells **in order**, with the **`Python 3.11 (heart_gpu)`** kernel. Register it once with `bash scripts/check_heart_env.sh --fix`.
* Every check prints `PASS`, `FAIL` or `WARN` and is recorded in `RESULTS`. The final cell lists all of them. A `FAIL` never stops the notebook; read it and fix it before continuing.
* Each code cell starts with a label: **[CPU]** runs anywhere, including the login node. **[GPU optional]** uses a GPU if this kernel has one, otherwise the CPU. **[SLURM]** submits or inspects a batch job.
* The notebook trains nothing long and installs nothing. Training runs under SLURM (`scripts/pretrain_smoke.sh`), not in this kernel.
* The real ligand LMDBs are opened read-only (`readonly=True, lock=False`), and only a few records are read.

| Part | What it checks |
|---|---|
| A | `model_train` filesystem: three primary directories, real LMDB files, output permissions |
| B | Kernel interpreter is `heart_gpu`; package versions and import paths; repair command |
| C | Uni-Core import, `unicore-train`, task/model/loss registry, real model and parameter count |
| D | A few real LMDB records; dictionary; 2D/3D paired-view pipeline and padding |
| E | Small real-data forward, loss and backward; gradients, NaN/Inf, GPU memory |
| F | Submit the 10-update smoke job with `sbatch`; job ID, status, logs |
| G | Losses, optimizer step count, `checkpoint_last.pt`, reload and resume, fatal errors; final verdict |

## 0. Setup

Paths follow the fixed CMRI layout. Override any of them with environment variables (`HEART_BASE`, `HEART_REPO`, `HEART_DATA_PATH`, `HEART_OUTPUT_ROOT`, `HEART_CONDA_ENV`, `HEART_UNICORE_SRC`) **before** you start the kernel.

In [ ]:
# [CPU] paths and PASS/FAIL bookkeeping
import os, sys, json, glob, re, time, shutil, subprocess
from pathlib import Path

BASE = Path(os.environ.get("HEART_BASE", "/home/ypan/heart_liver/model_train"))
REPO = Path(os.environ.get("HEART_REPO", BASE / "heart-molecular-pretraining"))
MODEL_ROOT = REPO / "model"
PRETRAIN_DIR = MODEL_ROOT / "pretrain"
UNICORE_SRC = Path(os.environ.get("HEART_UNICORE_SRC", BASE / "third_party" / "Uni-Core"))
DATA_PATH = Path(os.environ.get("HEART_DATA_PATH", BASE / "UniMolReference" / "ligand_data" / "ligands"))
OUTPUT_ROOT = Path(os.environ.get("HEART_OUTPUT_ROOT", REPO / "runs"))
SLURM_LOG_DIR = OUTPUT_ROOT / "slurm"
CONDA_ENV = Path(os.environ.get("HEART_CONDA_ENV", "/home/ypan/.conda/envs/heart_gpu"))
UNICORE_COMMIT = "ace6fae1c8479a9751f2bb1e1d6e4047427bc134"
EXPECTED_ENTRIES = {"train": int(os.environ.get("HEART_EXPECTED_TRAIN", 18837320)),
                    "valid": int(os.environ.get("HEART_EXPECTED_VALID", 98844))}
EXPECTED_PARAMS = 48144227          # unimol_contrastive_base, 64 heads (IMPLEMENTATION_MAP.md)

RESULTS = {}                        # "A1 name" -> (status, detail)

def check(key, ok, detail="", warn_only=False):
    status = "PASS" if ok else ("WARN" if warn_only else "FAIL")
    RESULTS[key] = (status, str(detail))
    print("{:4s}  {}{}".format(status, key, ("  -- " + str(detail)) if detail else ""))
    return ok

def run(cmd, cwd=None, timeout=600, show=True):
    """Run a command (list), print its output, return (returncode, output)."""
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=timeout)
        rc, out = p.returncode, (p.stdout or "") + (p.stderr or "")
    except FileNotFoundError:
        rc, out = 127, "command not found: {} (not on PATH in this kernel)".format(cmd[0])
    if show:
        print("$", " ".join(str(c) for c in cmd)); print(out.rstrip())
    return rc, out

for name in ("BASE", "REPO", "MODEL_ROOT", "UNICORE_SRC", "DATA_PATH", "OUTPUT_ROOT", "CONDA_ENV"):
    print("{:12s} = {}".format(name, globals()[name]))

## Part A — `model_train` filesystem

There must be exactly three primary directories under `BASE`: the HEART repository, `third_party/Uni-Core` and `UniMolReference`. Nothing is created here except a temporary write probe in `runs/`.

In [ ]:
# [CPU] A1: the three primary directories
for key, path in (("A1 repository", REPO), ("A1 Uni-Core checkout", UNICORE_SRC), ("A1 ligand data dir", DATA_PATH)):
    check(key, path.is_dir(), path)
print("\nentries in BASE:", sorted(p.name for p in BASE.iterdir()) if BASE.is_dir() else "BASE missing")

In [ ]:
# [CPU] A2: repository layout (unimol/ is a sibling of pretrain/)
for sub in ("pretrain/scripts", "pretrain/configs", "pretrain/tools", "pretrain/tests", "pretrain/notebooks",
            "unimol/data", "unimol/tasks", "unimol/models", "unimol/losses"):
    check("A2 model/" + sub, (MODEL_ROOT / sub).is_dir())
check("A2 no nested pretrain/unimol", not (PRETRAIN_DIR / "unimol").exists())

In [ ]:
# [CPU] A3: git commit of the code being tested
rc, out = run(["git", "-C", str(REPO), "log", "-1", "--format=%H %cd %s"])
check("A3 git commit readable", rc == 0, out.strip()[:80])
rc, out = run(["git", "-C", str(REPO), "status", "--short"])
check("A3 working tree clean", rc == 0 and not out.strip(), "local modifications present" if out.strip() else "", warn_only=True)

In [ ]:
# [CPU] A4: real LMDB files and dictionary (stat only, nothing is opened for writing)
for f in ("train.lmdb", "valid.lmdb", "dict.txt"):
    p = DATA_PATH / f
    ok = p.is_file() and os.access(p, os.R_OK)
    check("A4 readable " + f, ok, "{:.2f} GB".format(p.stat().st_size / 1e9) if ok else p)
if os.access(DATA_PATH, os.W_OK):
    print("note: the data directory is writable by you; HEART opens it read-only only")

In [ ]:
# [CPU] A5: output permissions (runs/ and runs/slurm/ must exist and be writable before sbatch)
for key, d in (("A5 runs/ writable", OUTPUT_ROOT), ("A5 runs/slurm/ writable", SLURM_LOG_DIR)):
    ok = d.is_dir()
    if ok:
        probe = d / ".heart_nb_write_test"
        try:
            probe.write_text("ok"); probe.unlink()
        except OSError as exc:
            ok = False
    check(key, ok, d if ok else "{} -> mkdir -p {}".format(d, d))
usage = shutil.disk_usage(OUTPUT_ROOT if OUTPUT_ROOT.exists() else REPO)
print("free disk: {:.0f} GB of {:.0f} GB".format(usage.free / 1e9, usage.total / 1e9))

## Part B — kernel and packages

The kernel must be the `heart_gpu` interpreter, with `~/.local` packages hidden (`PYTHONNOUSERSITE=1`). If this part fails, do not install anything from the notebook. In a terminal, run:

```bash
cd /home/ypan/heart_liver/model_train/heart-molecular-pretraining/model/pretrain
bash scripts/check_heart_env.sh --fix     # repair once (only missing/incompatible packages)
bash scripts/check_heart_env.sh           # must end with ALL CHECKS PASSED
```

Then restart this kernel and pick **Python 3.11 (heart_gpu)**.

In [ ]:
# [CPU] B1: which interpreter is this kernel?
import site
print("sys.executable:", sys.executable)
print("sys.prefix    :", sys.prefix)
print("python        :", sys.version.split()[0])
check("B1 kernel uses heart_gpu", Path(sys.prefix).resolve() == CONDA_ENV.resolve(), sys.prefix)
check("B1 Python 3.11", sys.version_info[:2] == (3, 11), sys.version.split()[0])
check("B1 user site-packages disabled", not site.ENABLE_USER_SITE,
      "start the kernel with PYTHONNOUSERSITE=1 (the heart_gpu kernelspec sets it)")
local = [p for p in sys.path if ".local" in p]
check("B1 no ~/.local on sys.path", not local, local)

In [ ]:
# [CPU] B2: make the env's bin/ first on PATH for subprocesses (unicore-train, pip)
env_bin = str(CONDA_ENV / ("Scripts" if os.name == "nt" else "bin"))
if not os.environ.get("PATH", "").startswith(env_bin):
    os.environ["PATH"] = env_bin + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
print("PATH[0] =", os.environ["PATH"].split(os.pathsep)[0])

In [ ]:
# [CPU] B3: versions and import paths of every dependency (check only, same tool as check_heart_env.sh)
rc, out = run([sys.executable, str(PRETRAIN_DIR / "tools" / "heart_env_check.py"),
               "--env-prefix", str(CONDA_ENV), "--requirements", str(PRETRAIN_DIR / "requirements-heart.txt"),
               "--unicore-src", str(UNICORE_SRC), "--unicore-commit", UNICORE_COMMIT])
check("B3 dependencies (heart_env_check.py)", rc == 0, "exit {} (2 = blocker such as torch/CUDA build)".format(rc))

In [ ]:
# [CPU] B4: torch build (never reinstalled by HEART tooling)
import torch
print("torch", torch.__version__, "| CUDA build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
check("B4 torch 2.2.2", torch.__version__.split("+")[0] == "2.2.2", torch.__version__)
check("B4 torch CUDA build 12.1", torch.version.cuda == "12.1", torch.version.cuda)
HAS_GPU = torch.cuda.is_available()
print("GPU visible to this kernel:", HAS_GPU, "(Parts C-E run on CPU when False)")

In [ ]:
# [CPU] B5: the repair command (printed, NOT executed)
print("cd {} && bash scripts/check_heart_env.sh --fix".format(PRETRAIN_DIR))

## Part C — Uni-Core, `unicore-train` and the HEART registry

`unicore-train` runs with `cwd = model/` and `--user-dir ./unimol`. This part imports the user module the same way.

In [ ]:
# [CPU] C1: Uni-Core import
import unicore
print("unicore", getattr(unicore, "__version__", "?"), unicore.__file__)
check("C1 unicore inside heart_gpu", Path(unicore.__file__).resolve().is_relative_to(CONDA_ENV.resolve()), unicore.__file__)

In [ ]:
# [CPU] C2: unicore-train entry point
exe = shutil.which("unicore-train")
print("unicore-train ->", exe)
check("C2 unicore-train inside heart_gpu", bool(exe) and Path(exe).resolve().is_relative_to(CONDA_ENV.resolve()), exe)

In [ ]:
# [CPU] C3: task / model / arch / loss registration through --user-dir ./unimol
from types import SimpleNamespace
from unicore import utils as uc_utils
from unicore.tasks import TASK_REGISTRY
from unicore.models import MODEL_REGISTRY, ARCH_MODEL_REGISTRY
from unicore.losses import CRITERION_REGISTRY
os.chdir(MODEL_ROOT)
uc_utils.import_user_module(SimpleNamespace(user_dir="./unimol"))
import unimol
check("C3 unimol resolves to model/unimol", Path(unimol.__file__).resolve().parent == (MODEL_ROOT / "unimol").resolve(), unimol.__file__)
check("C3 task unimol_contrastive", "unimol_contrastive" in TASK_REGISTRY)
check("C3 model unimol_contrastive", "unimol_contrastive" in MODEL_REGISTRY)
check("C3 arch unimol_contrastive_base", "unimol_contrastive_base" in ARCH_MODEL_REGISTRY)
check("C3 loss unimol_contrastive", "unimol_contrastive" in CRITERION_REGISTRY)

In [ ]:
# [CPU] C4: parse the smoke-test arguments with Uni-Core's own parser (values from configs/pilot.yaml,
#           which tests/test_scripts.py keeps identical to scripts/pretrain_smoke.sh)
sys.path.insert(0, str(PRETRAIN_DIR / "tools"))
import yaml_to_env
from unicore import options, tasks
launcher_cfg, uc_cfg = yaml_to_env.load(str(PRETRAIN_DIR / "configs" / "pilot.yaml"), [])
cli = yaml_to_env.to_cli(uc_cfg)
if not HAS_GPU:
    cli = [a for a in cli if a not in ("--fp16",)] + ["--cpu"]
parser = options.get_training_parser()
ARGS = options.parse_args_and_arch(parser, [str(DATA_PATH), "--user-dir", "./unimol"] + cli)
print("task/arch/loss:", ARGS.task, ARGS.arch, ARGS.loss)
print("layers {} | embed {} | ffn {} | heads {} | view mode {}".format(
    ARGS.encoder_layers, ARGS.encoder_embed_dim, ARGS.encoder_ffn_embed_dim, ARGS.encoder_attention_heads, ARGS.contrastive_view_mode))
check("C4 arguments parsed", ARGS.arch == "unimol_contrastive_base")

In [ ]:
# [CPU] C5: instantiate the actual model (random initialisation) and count parameters
TASK = tasks.setup_task(ARGS)              # reads dict.txt only
MODEL = TASK.build_model(ARGS)
LOSS = TASK.build_loss(ARGS)
n_params = sum(p.numel() for p in MODEL.parameters())
print("parameters: {:,}".format(n_params))
check("C5 parameter count {:,}".format(EXPECTED_PARAMS), n_params == EXPECTED_PARAMS, "{:,}".format(n_params))

## Part D — real LMDB records and the paired-view pipeline

Only a handful of records are decoded. The LMDB length comes from `txn.stat()` (patch P1), so nothing scans the 18.8M-record file.

In [ ]:
# [CPU] D1: entry counts and key convention (read-only)
import lmdb, pickle
def open_ro(path):
    return lmdb.open(str(path), subdir=False, readonly=True, lock=False, readahead=False, meminit=False, max_readers=256)
N = {}
for split in ("train", "valid"):
    env = open_ro(DATA_PATH / (split + ".lmdb"))
    with env.begin() as txn:
        N[split] = txn.stat()["entries"]
        first_last = txn.get(b"0") is not None and txn.get(str(N[split] - 1).encode()) is not None
    env.close()
    check("D1 {} entries == {:,}".format(split, EXPECTED_ENTRIES[split]), N[split] == EXPECTED_ENTRIES[split], "{:,}".format(N[split]))
    check("D1 {} keys b'0'..b'N-1' present".format(split), first_last)

In [ ]:
# [CPU] D2: decode three records per split: SMILES, atoms, conformers
import numpy as np
RECORDS = {}
for split in ("train", "valid"):
    env = open_ro(DATA_PATH / (split + ".lmdb"))
    with env.begin() as txn:
        for idx in (0, N[split] // 2, N[split] - 1):
            rec = pickle.loads(txn.get(str(idx).encode()))
            RECORDS[(split, idx)] = rec
            coords = rec["coordinates"]
            shapes = {np.asarray(c).shape for c in coords}
            finite = all(np.isfinite(np.asarray(c)).all() for c in coords)
            print("{}[{}] smi={} | atoms={} | conformers={} {} | finite={}".format(
                split, idx, rec.get("smi"), len(rec["atoms"]), len(coords), shapes, finite))
            check("D2 {}[{}] conformers (n_atoms,3) finite".format(split, idx),
                  finite and shapes == {(len(rec["atoms"]), 3)})
    env.close()
print("record keys:", sorted(next(iter(RECORDS.values())).keys()))

In [ ]:
# [CPU] D3: dictionary covers the elements seen
DICT = TASK.dictionary
print("dictionary size (incl. [MASK]):", len(DICT))
seen = sorted({a for rec in RECORDS.values() for a in rec["atoms"]})
missing = [a for a in seen if DICT.index(a) == DICT.unk()]
print("elements in sampled records:", seen)
check("D3 no sampled element maps to [UNK]", not missing, missing)

In [ ]:
# [CPU] D4: 3D conformers + RDKit 2D conformer (Add2DConformerDataset, used in train mode)
from unimol.data import LMDBDataset, Add2DConformerDataset
raw = LMDBDataset(str(DATA_PATH / "valid.lmdb"))
with2d = Add2DConformerDataset(raw, "smi", "atoms", "coordinates")
item_raw, item_2d = raw[0], with2d[0]
n3d, nall = len(item_raw["coordinates"]), len(item_2d["coordinates"])
z = np.asarray(item_2d["coordinates"][-1])[:, 2]
print("3D conformers: {} | after Add2D: {} | last conformer max|z| = {:.3g}".format(n3d, nall, np.abs(z).max()))
check("D4 2D conformer appended", nall == n3d + 1)
check("D4 appended conformer is planar (2D)", np.abs(z).max() < 1e-6)
check("D4 LMDB record not mutated (patch P2)", len(raw[0]["coordinates"]) == n3d)

In [ ]:
# [CPU] D5: full HEART pipeline on real valid records; two independent views per molecule
TASK.load_dataset("valid")
DS = TASK.dataset("valid")
DS.set_epoch(1)
print("dataset length:", len(DS))
s0 = DS[0]
v0, v1 = s0["view_0"], s0["view_1"]
same_tokens = all(np.array_equal(np.asarray(v0[k]), np.asarray(v1[k])) for k in v0 if k.endswith("src_tokens"))
same_coords = all(np.allclose(np.asarray(v0[k]), np.asarray(v1[k])) for k in v0 if k.endswith("src_coord"))
print("view keys:", list(v0.keys())[:6], "...")
check("D5 pipeline length == valid entries", len(DS) == N["valid"])
check("D5 views differ (independent mode)", not (same_tokens and same_coords),
      "identical views: contrastive-view-mode is not 'independent'" if same_tokens and same_coords else "")

In [ ]:
# [CPU] D6: collate a batch of 4 molecules -> 8 sequences (view_0 then view_1); padding
import torch
B = min(4, len(DS))   # 4 molecules on the real data
BATCH = DS.collater([DS[i] for i in range(B)])
toks = BATCH["net_input"]["src_tokens"]
pad = DICT.pad()
for k, v in BATCH["net_input"].items():
    print("net_input.{:14s} {}".format(k, tuple(v.shape)))
names = list(BATCH["target"]["smi_name"])
check("D6 batch holds 2 x {} sequences".format(B), toks.size(0) == 2 * B, tuple(toks.shape))
check("D6 view pairs aligned (same SMILES)", names[:B] == names[B:])
check("D6 padded length multiple of 8", toks.size(1) % 8 == 0, toks.size(1))
check("D6 padding present and consistent", bool(toks.eq(pad).any()) or B == 1,
      "pad tokens: {}".format(int(toks.eq(pad).sum())), warn_only=True)
check("D6 coordinates finite", bool(torch.isfinite(BATCH["net_input"]["src_coord"]).all()))
check("D6 masked atoms present", int(BATCH["target"]["tokens_target"].ne(pad).sum()) > 0)

## Part E — small forward / loss / backward on real data

This uses the batch from D6 and the model from C5, in fp32, for one forward and backward pass. **No optimizer step is taken.** The cells run on the GPU if this kernel has one, and on the CPU otherwise (about a minute on CPU). This is a functional check only. The fp16 training path is exercised by the smoke job (Part F).

In [ ]:
# [GPU optional] E1: move model and batch to the device
from unicore import utils as uc_utils
DEVICE = torch.device("cuda" if HAS_GPU else "cpu")
MODEL = MODEL.to(DEVICE).train()
sample = uc_utils.move_to_cuda(BATCH) if HAS_GPU else BATCH
if HAS_GPU:
    torch.cuda.reset_peak_memory_stats()
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none")

In [ ]:
# [GPU optional] E2: forward + loss (the registered unimol_contrastive loss)
t0 = time.time()
loss, sample_size, logging_output = LOSS(MODEL, sample)
print("forward+loss {:.1f}s | loss {:.4f} | sample_size {}".format(time.time() - t0, float(loss), sample_size))
for k in ("masked_token_loss", "masked_coord_loss", "masked_dist_loss", "contrastive_loss",
          "x_norm_loss", "delta_pair_repr_norm_loss"):
    if k in logging_output:
        print("  {:28s} {}".format(k, float(logging_output[k])))
check("E2 loss finite", bool(torch.isfinite(loss)))
check("E2 contrastive term logged", "contrastive_loss" in logging_output)

In [ ]:
# [GPU optional] E3: backward; gradients present and finite
MODEL.zero_grad(set_to_none=True)
loss.backward()
grads = [p.grad for p in MODEL.parameters() if p.requires_grad]
n_with = sum(g is not None for g in grads)
finite = all(torch.isfinite(g).all() for g in grads if g is not None)
gnorm = torch.norm(torch.stack([g.detach().float().norm() for g in grads if g is not None]))
print("params with grad: {}/{} | grad norm {:.4g}".format(n_with, len(grads), float(gnorm)))
check("E3 gradients finite (no NaN/Inf)", finite)
check("E3 most parameters receive gradients", n_with >= 0.9 * len(grads), "{}/{}".format(n_with, len(grads)))

In [ ]:
# [GPU optional] E4: memory (GPU only) and cleanup
if HAS_GPU:
    peak = torch.cuda.max_memory_allocated() / 2**30
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print("peak allocated {:.2f} GiB of {:.1f} GiB (fp32, batch {})".format(peak, total, B))
    check("E4 fits in GPU memory", peak < 0.9 * total)
else:
    print("no GPU in this kernel: memory check skipped; the smoke job reports gb_free per update")
MODEL.zero_grad(set_to_none=True)
del loss
if HAS_GPU:
    torch.cuda.empty_cache()

In [ ]:
# [CPU] E5: Parts A-E summary -- all must PASS before submitting the smoke job
fails = [k for k, (s, _) in RESULTS.items() if s == "FAIL"]
print("checks: {} | FAIL: {}".format(len(RESULTS), len(fails)))
for k in fails:
    print("  FAIL", k, "--", RESULTS[k][1])
READY_FOR_SMOKE = not fails
print("READY_FOR_SMOKE =", READY_FOR_SMOKE)

## Part F — submit the 30-minute, 10-update GPU smoke job

`scripts/pretrain_smoke.sh` requests 1 GPU, 6 CPUs, 55G and 00:30:00 on partition `gpu`. It runs 10 genuine fp16 optimizer updates on the real LMDBs, validates at updates 5 and 10, saves checkpoints and checks them. It never starts full pretraining.

Set `SUBMIT_SMOKE = True` in the next cell to submit. The job runs under SLURM; this kernel only watches it.

In [ ]:
# [SLURM] F1: submit (guarded)
SUBMIT_SMOKE = False          # <- set True deliberately
BATCH_SIZE_OVERRIDE = None    # e.g. 2 after a documented T4 OOM (then 1); None = pilot value 4
JOB_FILE = SLURM_LOG_DIR / "last_smoke_job.txt"
if SUBMIT_SMOKE:
    assert READY_FOR_SMOKE, "Parts A-E have FAIL items; fix them first"
    env = dict(os.environ)
    if BATCH_SIZE_OVERRIDE:
        env["BATCH_SIZE"] = str(BATCH_SIZE_OVERRIDE)
    p = subprocess.run(["sbatch", "--parsable", "scripts/pretrain_smoke.sh"], cwd=PRETRAIN_DIR,
                       capture_output=True, text=True, env=env)
    print(p.stdout, p.stderr)
    JOB_ID = p.stdout.strip().split(";")[0]
    check("F1 sbatch accepted", p.returncode == 0 and JOB_ID.isdigit(), JOB_ID)
    JOB_FILE.write_text(JOB_ID + "\n")
else:
    JOB_ID = JOB_FILE.read_text().strip() if JOB_FILE.exists() else None
    print("not submitting; last recorded smoke job:", JOB_ID)

In [ ]:
# [SLURM] F2: status (squeue while queued/running, sacct afterwards)
def job_state(job_id):
    rc, out = run(["sacct", "-j", str(job_id), "-X", "-n", "-P", "--format=State,Elapsed,ExitCode"], show=False)
    if rc == 127:
        return "NO_SACCT"
    return out.strip().splitlines()[0] if rc == 0 and out.strip() else "UNKNOWN"
if JOB_ID:
    run(["squeue", "-j", str(JOB_ID), "-o", "%.10i %.20j %.8T %.10M %.10l %R"])
    print("sacct:", job_state(JOB_ID))

In [ ]:
# [SLURM] F3: wait for the job to finish (polls every 30 s, gives up after 40 min)
if JOB_ID:
    t_end = time.time() + 40 * 60
    while time.time() < t_end:
        state = job_state(JOB_ID)
        if state.split("|")[0] in ("COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "OUT_OF_MEMORY", "NODE_FAIL", "NO_SACCT"):
            break
        time.sleep(30)
    print("final:", state)
    check("F3 job COMPLETED (exit 0:0)", state.startswith("COMPLETED|") and state.endswith("|0:0"), state)

In [ ]:
# [SLURM] F4: SLURM stdout/stderr (actual log paths)
OUT_LOG = SLURM_LOG_DIR / "heart-smoke-test-{}.out".format(JOB_ID)
ERR_LOG = SLURM_LOG_DIR / "heart-smoke-test-{}.err".format(JOB_ID)
for f in (OUT_LOG, ERR_LOG):
    print("==== {} ({})".format(f, "exists" if f.exists() else "missing"))
    if f.exists():
        print("".join(f.read_text(errors="replace").splitlines(True)[-40:]))

## Part G — evidence review

A PASS here requires evidence from the job's own files.

In [ ]:
# [CPU] G1: locate the run directory from the job log
RUN_DIR = None
if not JOB_ID:
    print("no smoke job recorded yet: run Part F first (G1-G4 record nothing; G7 reports NOT RUN)")
elif OUT_LOG.exists():
    m = re.findall(r"run dir: (\S+)", OUT_LOG.read_text(errors="replace"))
    RUN_DIR = Path(m[-1]) if m else None
print("run dir:", RUN_DIR)
if JOB_ID:
    check("G1 run directory exists", bool(RUN_DIR) and RUN_DIR.is_dir(), RUN_DIR)
if RUN_DIR and RUN_DIR.is_dir():
    for p in sorted(RUN_DIR.iterdir()):
        print("  {:40s} {:>10.1f} MB".format(p.name, p.stat().st_size / 2**20 if p.is_file() else 0))

In [ ]:
# [CPU] G2: train / valid losses and the optimizer step count
TRAIN_LOG = RUN_DIR / "train-{}.log".format(JOB_ID) if RUN_DIR else None
if TRAIN_LOG and TRAIN_LOG.exists():
    rc, out = run([sys.executable, str(PRETRAIN_DIR / "tools" / "summarize_log.py"), str(TRAIN_LOG), "--expect-updates", "10"])
    check("G2 10 updates, finite train+valid losses, checkpoint saved", rc == 0)
elif JOB_ID:
    check("G2 training log present", False, TRAIN_LOG)

In [ ]:
# [CPU] G3: checkpoint_last.pt records 10 optimizer updates and reloads strictly
CKPT = RUN_DIR / "checkpoint_last.pt" if RUN_DIR else None
if CKPT and CKPT.exists():
    rc, out = run([sys.executable, str(PRETRAIN_DIR / "tools" / "verify_checkpoint.py"), str(CKPT),
                   "--expect-updates", "10", "--data", str(DATA_PATH)], cwd=MODEL_ROOT)
    check("G3 checkpoint_last.pt @ 10 updates, reloadable", rc == 0)
elif JOB_ID:
    check("G3 checkpoint_last.pt exists", False, CKPT)

In [ ]:
# [CPU] G4: fatal errors in the SLURM logs (CUDA OOM, DataLoader worker crash, Traceback)
patterns = {"CUDA out of memory": "OOM -> resubmit with BATCH_SIZE=2 (then 1) and record it",
            "DataLoader worker": "data loader crash", "Traceback": "Python exception", "Killed": "host OOM / killed"}
if JOB_ID:
    text = "".join(f.read_text(errors="replace") for f in (OUT_LOG, ERR_LOG) if f.exists())
    hits = {p: why for p, why in patterns.items() if p in text}
    check("G4 no fatal CUDA OOM / data-loader / Python errors", bool(text) and not hits, hits or "")
    VERDICT_FILE = RUN_DIR / "smoke_result-{}.txt".format(JOB_ID) if RUN_DIR else None
    ok = bool(VERDICT_FILE) and VERDICT_FILE.exists()
    if ok:
        print(VERDICT_FILE.read_text())
    check("G4 script verdict file says PASS", ok and "verdict=PASS" in VERDICT_FILE.read_text(), VERDICT_FILE)

### Resume test (continue the same run from update 10 to 12)

This submits a second short job, about 10 minutes, with `RESUME_DIR` and `MAX_UPDATE=12`. Uni-Core restores the model, optimizer, LR schedule, fp16 scaler and data position from `checkpoint_last.pt`.

In [ ]:
# [SLURM] G5: submit the resume test (guarded)
SUBMIT_RESUME = False         # <- set True deliberately after G1-G4 PASS
RESUME_FILE = SLURM_LOG_DIR / "last_smoke_resume_job.txt"
if SUBMIT_RESUME:
    env = dict(os.environ, RESUME_DIR=str(RUN_DIR), MAX_UPDATE="12")
    p = subprocess.run(["sbatch", "--parsable", "--job-name=heart-smoke-resume", "scripts/pretrain_smoke.sh"],
                       cwd=PRETRAIN_DIR, capture_output=True, text=True, env=env)
    print(p.stdout, p.stderr)
    RESUME_JOB = p.stdout.strip().split(";")[0]
    RESUME_FILE.write_text(RESUME_JOB + "\n")
else:
    RESUME_JOB = RESUME_FILE.read_text().strip() if RESUME_FILE.exists() else None
print("resume job:", RESUME_JOB)

In [ ]:
# [CPU] G6: resume evidence (run after the resume job has finished; see F2/F3 with JOB_ID=RESUME_JOB)
if RESUME_JOB and RUN_DIR:
    print("sacct:", job_state(RESUME_JOB))
    rlog = RUN_DIR / "train-{}.log".format(RESUME_JOB)
    if rlog.exists():
        rc, _ = run([sys.executable, str(PRETRAIN_DIR / "tools" / "summarize_log.py"), str(rlog),
                     "--expect-updates", "12", "--expect-resume-from", "10"])
        check("G6 resumed @ 10 and reached 12 updates", rc == 0)
        rc, _ = run([sys.executable, str(PRETRAIN_DIR / "tools" / "verify_checkpoint.py"), str(RUN_DIR / "checkpoint_last.pt"),
                     "--expect-updates", "12", "--data", str(DATA_PATH)], cwd=MODEL_ROOT)
        check("G6 checkpoint_last.pt @ 12 updates", rc == 0)
    else:
        print("resume log not there yet:", rlog)

In [ ]:
# [CPU] G7: final verdict, from recorded evidence only
print("{:5s} {}".format("", "check"))
for k, (s, d) in RESULTS.items():
    print("{:5s} {}{}".format(s, k, ("  -- " + d) if d and s != "PASS" else ""))
required = [k for k in RESULTS if k.startswith(("A", "B", "C", "D", "E", "F", "G"))]
smoke_keys = [k for k in RESULTS if k.startswith(("F3", "G1", "G2", "G3", "G4"))]
fails = [k for k in required if RESULTS[k][0] == "FAIL"]
if not JOB_ID or not smoke_keys:
    print("\nSMOKE TEST: NOT RUN (no SLURM evidence in this kernel)")
elif fails:
    print("\nSMOKE TEST: FAIL ({} failing checks)".format(len(fails)))
else:
    print("\nSMOKE TEST: PASS ({} checks{})".format(len(required), "" if any(k.startswith("G6") for k in RESULTS) else "; resume test not yet reviewed"))

## Full-training gate (separate, manual decision)

Full pretraining (`scripts/pretrain_heart.sh`) is never submitted from this notebook. Before anyone submits it:

1. The smoke test above is **PASS**, including the resume test (G6).
2. **VRAM probe** on the target GPU (an H200 is recommended; a 1M-update run on a T4 is not practical). Run short smoke jobs with growing micro-batches, for example `BATCH_SIZE=64/128/256/448 MAX_UPDATE=20 sbatch --gres=gpu:<h200-type>:1 --job-name=heart-probe-bsN scripts/pretrain_smoke.sh`. Pick the largest micro-batch that leaves several GiB free (`gb_free` in the log) with no OOM.
3. Write down the micro-batch (per GPU), `UPDATE_FREQ` (accumulation) and the effective batch = micro-batch × update-freq × GPUs. LUMI used 448 per GPU. With accumulation, NT-Xent negatives still come from a single micro-batch.
4. Submit by hand, with explicit confirmation, from `model/pretrain`:

```bash
HEART_CONFIRM_FULL=1 BATCH_SIZE=<probed> UPDATE_FREQ=1 HEART_SMOKE_RUN=<smoke run dir> \
    sbatch --gres=gpu:<h200-type>:1 scripts/pretrain_heart.sh
```

The run directory is fixed (`runs/full/heart-pretrain-full-v1`). The job stops cleanly 30 minutes before the time limit, and resubmitting the same command resumes from `checkpoint_last.pt`.